# M1-B2 — Classifieur de motifs d'insatisfaction

Une section par tâche du brief. Les cellules `TODO` sont à compléter.

In [20]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # avant tout « import keras » : Keras utilisera PyTorch

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
CACHE = DATA / "cache"
CACHE.mkdir(exist_ok=True)
MODELS = ROOT / "models"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Tâche 1 — Démarrer

Cette cellule télécharge le modèle d'embeddings au premier lancement (environ 470 Mo) 

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(MODEL_NAME)
vecteurs = encoder.encode(["Le lit était très inconfortable.", "Le matelas était défoncé."])
vecteurs.shape   # (2, 384) : deux phrases, 384 nombres chacune

## Tâche 2 — Explorer le corpus

In [ ]:
avis = pd.read_csv(DATA / "avis_motifs.csv")
recents = pd.read_csv(DATA / "avis_recents.csv")
avis.head()

In [ ]:
avis["longueur_texte"] = avis["texte"].str.len()
avis_par_motif = avis.sort_values(["motif", "avis_id"])

pd.set_option("display.max_colwidth", 200)
echantillon = avis.sample(20, random_state=0).sort_values([ "avis_id","motif"])
display(echantillon[["avis_id", "texte","motif",  "note", "longueur_texte"]])

### Notre avis sur les étiquettes du LLM

| avis_id | Étiquette | Observation | Étiquette proposée|
|---|---|---|---|
| 444 | `accueil` | Manque de rangements et de cintres | `chambre` ou `equipements` |
| 205 | `bruit` | Chambre humide, sans fenêtre et avec peu de rangements ; aucun bruit mentionné | `chambre` |
| 988 | `restauration` | Critique du personnel, sans mention de nourriture | `accueil` |

## Tâche 3 — Une référence : TF-IDF + régression logistique

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score

train, test = train_test_split(avis, test_size=0.25, stratify=avis["motif"], random_state=42)

def f1_macro(y_true, y_pred):
    return round(f1_score(y_true, y_pred, average="macro"), 3)

tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
tfidf.fit(train["texte"], train["motif"])
print("TF-IDF — F1 macro test :", f1_macro(test["motif"], tfidf.predict(test["texte"])))


## Tâche 4 — Des embeddings

Encode les avis **une seule fois**, puis sauvegarde les vecteurs (par exemple `np.save(CACHE / "avis.npy", ...)`)
pour ne pas les recalculer à chaque essai.

In [ ]:
def encode_cached(textes, nom):
    chemin = CACHE / f"{nom}.npy"
    if chemin.exists():
        return np.load(chemin)
    vecteurs = encoder.encode(list(textes), batch_size=64, show_progress_bar=True)
    np.save(chemin, vecteurs)
    return vecteurs

E_avis = encode_cached(avis["texte"], "avis")          # une ligne par avis, dans l'ordre de `avis`
E_recents = encode_cached(recents["texte"], "recents")
E_avis.shape, E_recents.shape

In [ ]:
E_train = E_avis[train.index]
E_test = E_avis[test.index]

emb_lr = LogisticRegression(max_iter=3000).fit(E_train, train["motif"])
print("Embeddings + logreg — F1 macro test :", f1_macro(test["motif"], emb_lr.predict(E_test)))

In [ ]:
pd.DataFrame({
    "tfidf": f1_score(test["motif"], tfidf.predict(test["texte"]), average=None),
    "embeddings": f1_score(test["motif"], emb_lr.predict(E_test), average=None),
}, index=sorted(avis["motif"].unique())).round(2)

## Tâche 5 — Un petit réseau Keras

In [ ]:
#Note : les données préparées E_train et train proviennent des cellules précédentes

import keras
from sklearn.preprocessing import LabelEncoder


keras.utils.set_random_seed(42)
le = LabelEncoder().fit(avis["motif"])
y_train = le.transform(train["motif"])

reseau = keras.Sequential([
    keras.Input(shape=(E_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(le.classes_), activation="softmax"),
])
reseau.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
reseau.summary()

arret = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

historique = reseau.fit(
    E_train, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)
print("arrêt après", len(historique.history["loss"]), "époques")



In [ ]:
plt.plot(historique.history["loss"], label="entraînement")
plt.plot(historique.history["val_loss"], label="validation")
plt.xlabel("époque"); plt.ylabel("perte (log loss)"); plt.legend(); plt.show()


### ✏️ Que montrent tes courbes ? Pourquoi l'arrêt précoce s'est-il déclenché ?

*La perte d’entraînement diminue, tandis que la perte de validation finit par stagner ou remonter : le réseau apprend alors de moins en moins bien à généraliser et commence possiblement à surapprendre. L’arrêt précoce s’est déclenché après cinq époques sans amélioration de la perte de validation (`patience=5`). Les poids de l’époque où cette perte était la plus faible ont été restaurés.*

In [ ]:
def predict_reseau(E):
    return le.inverse_transform(reseau.predict(E, verbose=0).argmax(axis=1))

print("Keras — F1 macro test :", f1_macro(test["motif"], predict_reseau(E_test)))

## Tâche 6 — Comparer et analyser

In [ ]:
comparaison = pd.DataFrame([
    {"modèle": "TF-IDF + logreg",
     "F1 test": f1_macro(test["motif"], tfidf.predict(test["texte"])),
     "F1 récents": f1_macro(recents["motif"], tfidf.predict(recents["texte"]))},
    {"modèle": "embeddings + logreg",
     "F1 test": f1_macro(test["motif"], emb_lr.predict(E_test)),
     "F1 récents": f1_macro(recents["motif"], emb_lr.predict(E_recents))},
    {"modèle": "embeddings + Keras",
     "F1 test": f1_macro(test["motif"], predict_reseau(E_test)),
     "F1 récents": f1_macro(recents["motif"], predict_reseau(E_recents))},
])
comparaison

Le classement n’est pas le même : sur le jeu de test, TF-IDF + régression logistique obtient le meilleur F1 macro (0,911). Sur les avis récents, les embeddings + Keras arrivent en tête (0,682), devant les embeddings + régression logistique (0,662), tandis que TF-IDF chute à 0,389.

*TF-IDF s’appuie surtout sur les mots présents dans les exemples d’entraînement. Si les avis récents emploient un vocabulaire ou des formulations différents, les mots-clés appris correspondent moins bien. Les embeddings capturent davantage le sens et les similarités entre formulations, ce qui peut mieux résister à ce décalage.*

In [ ]:
ConfusionMatrixDisplay.from_predictions(recents["motif"], predict_reseau(E_recents), xticks_rotation=45)
plt.show()

In [ ]:
import joblib
import tempfile
import time

modeles = [
    ("TF-IDF + logreg", tfidf.predict, test["texte"], recents["texte"], tfidf),
    ("embeddings + logreg", emb_lr.predict, E_test, E_recents, emb_lr),
    ("embeddings + Keras", predict_reseau, E_test, E_recents, reseau),
]

predictions = {}
resultats = []
for nom, predire, X_test, X_recents, _ in modeles:
    debut = time.perf_counter()
    pred_test = predire(X_test)
    temps_test = time.perf_counter() - debut

    debut = time.perf_counter()
    pred_recents = predire(X_recents)
    temps_recents = time.perf_counter() - debut

    predictions[nom] = (pred_test, pred_recents)
    resultats.append({
        "modèle": nom,
        "F1 test": f1_macro(test["motif"], pred_test),
        "F1 récents": f1_macro(recents["motif"], pred_recents),
        "temps test (s)": round(temps_test, 4),
        "temps récents (s)": round(temps_recents, 4),
    })

# Mesure la taille des modèles sérialisés ; les fichiers temporaires sont supprimés ensuite.
with tempfile.TemporaryDirectory() as dossier_temporaire:
    tailles_ko = {}
    for nom, _, _, _, modele in modeles:
        if nom == "embeddings + Keras":
            chemin = Path(dossier_temporaire) / "reseau.keras"
            modele.save(chemin)
        else:
            chemin = Path(dossier_temporaire) / f"{nom}.joblib"
            joblib.dump(modele, chemin)
        tailles_ko[nom] = round(chemin.stat().st_size / 1024, 1)

evaluation = pd.DataFrame(resultats)
evaluation["taille modèle (Ko)"] = evaluation["modèle"].map(tailles_ko)
display(evaluation)

# Temps mesuré sur un lot, hors calcul des embeddings ; matrice vraie × prédite.
classes = sorted(avis["motif"].unique())
fig, axes = plt.subplots(len(modeles), 2, figsize=(14, 16))
for i, (nom, _, _, _, _) in enumerate(modeles):
    pred_test, pred_recents = predictions[nom]
    ConfusionMatrixDisplay.from_predictions(
        test["motif"], pred_test, labels=classes, xticks_rotation=45,
        cmap="Blues", colorbar=False, ax=axes[i, 0]
    )
    axes[i, 0].set_title(f"{nom} — test")
    ConfusionMatrixDisplay.from_predictions(
        recents["motif"], pred_recents, labels=classes, xticks_rotation=45,
        cmap="Blues", colorbar=False, ax=axes[i, 1]
    )
    axes[i, 1].set_title(f"{nom} — avis récents")
plt.tight_layout()
plt.show()

In [ ]:
pred_test_reseau = predict_reseau(E_test)
erreurs = test.assign(prediction=pred_test_reseau)
erreurs = erreurs.loc[erreurs["motif"] != erreurs["prediction"]]

print(f"{len(erreurs)} erreurs du réseau sur {len(test)} avis de test")
display(
    erreurs.head(10)[["avis_id", "texte", "motif", "prediction", "note"]]
    .rename(columns={"motif": "étiquette LLM"})
)

## 10 erreurs examinées

| avis_id | texte | étiquette LLM | prédiction | note | erreur du modèle ou étiquette douteuse ? |
|---|---|---|---|---:|---|
| 423 | L'emplacement est pourtant idéal. La salle de bain n'avait pas de porte, aucune intimité. À éviter. | restauration | chambre | 1 | Étiquette douteuse : la critique porte clairement sur la salle de bain/chambre. |
| 1171 | Lit beaucoup trop petit pour deux adultes. La réceptionniste passait plus de temps sur son téléphone qu'avec les clients. Aucune aide pour monter nos valises… | chambre | accueil | 2 | Avis ambigu : plusieurs motifs, lit ou chambre d'un côté, accueil de l'autre. |
| 882 | L'accès à l'hôtel est mal indiqué… Le minibar est hors de prix… Prix de la chambre doublé… | emplacement | prix | 2 | Étiquette douteuse : le prix est au moins aussi central que l'emplacement. |
| 389 | La salle de bain était propre. Le réceptionniste ne nous a même pas regardés. | accueil | proprete | 1 | Erreur du modèle : l'accueil est explicitement critiqué. |
| 155 | Le restaurant mérite le détour. Le concierge… Bruit des camions… Les portes claquent… | accueil | bruit | 1 | Étiquette douteuse : l'avis contient restauration, accueil et bruit. |
| 1163 | La literie était confortable. La carte magnétique se démagnétisait tous les jours. | equipements | chambre | 2 | Erreur du modèle : la carte magnétique relève davantage des équipements. |
| 867 | Vacances en famille… Prix… Pas assez de rangements, et pas un seul cintre. | chambre | prix | 1 | Avis ambigu : problèmes de chambre et de prix. |
| 1095 | Le minibar est hors de prix. Nous avons attendu 40 minutes à la réception… | accueil | prix | 2 | Étiquette raisonnable : l'attente à la réception est l'élément central. |
| 945 | buffet à volonté… qualité cantine… accueil glacial à la réception. | accueil | restauration | 2 | Avis ambigu : restauration et accueil sont tous deux fortement évoqués. |
| 1181 | La télévision ne captait aucune chaîne. Pas d'eau chaude le matin. | restauration | equipements | 2 | Étiquette manifestement douteuse : il s'agit d'équipements. |# 10 erreurs examinées


In [ ]:
MODELS.mkdir(parents=True, exist_ok=True)

reseau.save(MODELS / "classifieur_motifs.keras")
joblib.dump(le, MODELS / "label_encoder.joblib")

print("Modèle sauvegardé dans :", MODELS)